# Create runlist CSVs for the multi-SRA-per-pod weebill workflow

This notebook builds the CSV files that feed `sra_multi_workflow_template.yaml`.

Unlike the chunked workflow (`prod_workflow_template4.yaml` + `batch_creation 202602.ipynb`),
here each **pod processes several whole SRA runs** in a loop (like the Logan workflow),
with **no read chunking**. Per accession the pod streams `sracat-rs` reads straight into
`weebill sketch --merge` via FIFOs (no fastq temp files). To stop a node being swamped when
many download-heavy pods land on it at once, we pack accessions into pods with a size budget
and emit an `ephemeral_storage_mb` request per pod. The k8s scheduler bin-packs pods by that
request, so it will not overcommit a node's local disk.

Output CSV columns (one row = one pod):

| column | fed into template parameter | meaning |
| --- | --- | --- |
| `pod_accessions` | `SRA_accession_num` | space-separated list of accessions the pod loops over |
| `batch_name` | `batch_name` | S3 output prefix |
| `ephemeral_storage_mb` | `ephemeral_storage_mb` | requested local disk (MiB) = scheduler size constraint |
| `pod_id` | (naming only) | 0-based pod index within the batch |
| `n_accessions` | (info) | number of accessions in the pod |
| `total_mbytes` | (info) | summed SRA file size of the pod's accessions |

In [1]:
import polars as pl
import os

In [2]:
def show_all(df, width=200, max_col_width=True):
    '''Print an entire polars dataframe in the console or notebook output.'''
    with pl.Config() as cfg:
        cfg.set_tbl_cols(-1)
        cfg.set_tbl_rows(-1)
        cfg.set_tbl_width_chars(width)
        if max_col_width or len(df.columns) == 1:
            cfg.set_fmt_str_lengths(width)
        print(df)

## Gather the list of metagenomes we need to process

Same inputs as `batch_creation 202602.ipynb`: the full SRA metadata dump, minus everything
already processed in previous runs.

In [3]:
# Read in list of all the metagenomes we need to process
columns = ['acc','releasedate','organism','mbytes','avespotlen','bases','librarylayout']
all_metags = pl.read_csv('~/git/sandpiper/sra_metadata/sra_metadata_20260303.some_columns.csv.gz', has_header=False)
all_metags.columns = columns
all_metags.head(), all_metags.shape

(shape: (5, 7)
 ┌─────────────┬────────────────┬────────────────┬────────┬────────────┬────────────┬───────────────┐
 │ acc         ┆ releasedate    ┆ organism       ┆ mbytes ┆ avespotlen ┆ bases      ┆ librarylayout │
 │ ---         ┆ ---            ┆ ---            ┆ ---    ┆ ---        ┆ ---        ┆ ---           │
 │ str         ┆ str            ┆ str            ┆ i64    ┆ i64        ┆ i64        ┆ str           │
 ╞═════════════╪════════════════╪════════════════╪════════╪════════════╪════════════╪═══════════════╡
 │ ERR15880073 ┆ 2025-11-19T00: ┆ wastewater     ┆ 810    ┆ 216        ┆ 2651825214 ┆ PAIRED        │
 │             ┆ 00:00+00:00    ┆ metagenome     ┆        ┆            ┆            ┆               │
 │ SRR16203935 ┆ 2021-10-05T00: ┆ tick           ┆ 769    ┆ 302        ┆ 1825027978 ┆ PAIRED        │
 │             ┆ 00:00+00:00    ┆ metagenome     ┆        ┆            ┆            ┆               │
 │ SRR34723171 ┆ 2025-12-01T00: ┆ human          ┆ 2440   ┆ 302    

In [ ]:
# Read in list of metagenomes already processed
# prev = pl.read_csv('~/m/msingle/mess/193_202502_sra_scrape/find_gz20250402.accessions.uniq.txt', has_header=False)
# prev.columns = ['acc']
# prev.head(), prev.shape

In [4]:
# Subtract the two to get the list of metagenomes we still need to process
# to_process = all_metags.filter(~pl.col('acc').is_in(prev['acc']))
to_process = all_metags
to_process.head(), to_process.shape

(shape: (5, 7)
 ┌─────────────┬────────────────┬────────────────┬────────┬────────────┬────────────┬───────────────┐
 │ acc         ┆ releasedate    ┆ organism       ┆ mbytes ┆ avespotlen ┆ bases      ┆ librarylayout │
 │ ---         ┆ ---            ┆ ---            ┆ ---    ┆ ---        ┆ ---        ┆ ---           │
 │ str         ┆ str            ┆ str            ┆ i64    ┆ i64        ┆ i64        ┆ str           │
 ╞═════════════╪════════════════╪════════════════╪════════╪════════════╪════════════╪═══════════════╡
 │ ERR15880073 ┆ 2025-11-19T00: ┆ wastewater     ┆ 810    ┆ 216        ┆ 2651825214 ┆ PAIRED        │
 │             ┆ 00:00+00:00    ┆ metagenome     ┆        ┆            ┆            ┆               │
 │ SRR16203935 ┆ 2021-10-05T00: ┆ tick           ┆ 769    ┆ 302        ┆ 1825027978 ┆ PAIRED        │
 │             ┆ 00:00+00:00    ┆ metagenome     ┆        ┆            ┆            ┆               │
 │ SRR34723171 ┆ 2025-12-01T00: ┆ human          ┆ 2440   ┆ 302    

## Pack accessions into pods with a size budget

`mbytes` is the SRA download size. We:

1. Drop runs bigger than `MAX_SINGLE_RUN_MBYTES` (they belong in the chunked workflow, not
   this whole-run one, and would blow a single pod's disk).
2. First-fit-decreasing bin-pack the rest into pods, capping both the summed `mbytes`
   (`MAX_POD_MBYTES`) and the count (`MAX_ACCS_PER_POD`). Large runs end up ~alone; many
   small runs share a pod.
3. Set `ephemeral_storage_mb = max_mbytes * STORAGE_SAFETY_FACTOR + STORAGE_HEADROOM_MB`, where
   `max_mbytes` is the *largest single run* in the pod (not the summed `total_mbytes`).
   Accessions in a pod are processed one at a time in a loop, and sracat-rs streams reads
   straight into `weebill sketch` via FIFOs (no fastq temp files), so at most one `.sra` file
   is ever on disk; the small safety factor only covers filesystem overhead and the tiny
   `.sylspc` sketch outputs. This value is what the k8s scheduler uses to avoid overloading a
   node.

In [ ]:
# --- Packing / sizing parameters -------------------------------------------------
MAX_SINGLE_RUN_MBYTES = 30_000    # skip runs larger than this (~20 GB); use the chunked workflow instead
MAX_POD_MBYTES        = 30_000    # size budget per pod (summed download size)
MAX_ACCS_PER_POD      = 50       # cap accessions per pod so a pod is not too long-running
STORAGE_SAFETY_FACTOR = 1.3       # local disk vs raw .sra size. sracat-rs streams into weebill (no
                                 # fastq temp files), so ~1x .sra is enough; 2x is a safe margin.
STORAGE_HEADROOM_MB   = 5000     # fixed headroom (working files, .sylspc sketch outputs)


def pack_into_pods(df, batch_name,
                   max_pod_mbytes=MAX_POD_MBYTES,
                   max_accs_per_pod=MAX_ACCS_PER_POD,
                   max_single_run_mbytes=MAX_SINGLE_RUN_MBYTES,
                   storage_safety_factor=STORAGE_SAFETY_FACTOR,
                   storage_headroom_mb=STORAGE_HEADROOM_MB):
    '''First-fit-decreasing bin packing of accessions into pods by mbytes.

    df must have columns 'acc' and 'mbytes'. Returns one row per pod. The caps
    default to the globals above but can be overridden per call (e.g. a small
    max_accs_per_pod to force a test batch across several pods).
    '''
    runs = (
        df.select('acc', 'mbytes')
          .filter(pl.col('mbytes') <= max_single_run_mbytes)
          .sort('mbytes', descending=True)
    )
    print(f"{runs.height} runs to pack after dropping those > {max_single_run_mbytes} mbytes "
          f"({df.height - runs.height} dropped)")

    bins = []  # each bin: dict(accs=[...], total=int, max=int)
    for acc, mbytes in runs.iter_rows():
        placed = False
        for b in bins:
            if b['total'] + mbytes <= max_pod_mbytes and len(b['accs']) < max_accs_per_pod:
                b['accs'].append(acc)
                b['total'] += mbytes
                b['max'] = max(b['max'], mbytes)
                placed = True
                break
        if not placed:
            bins.append({'accs': [acc], 'total': mbytes, 'max': mbytes})

    pods = pl.DataFrame({
        'pod_id': list(range(len(bins))),
        'pod_accessions': [' '.join(b['accs']) for b in bins],
        'n_accessions': [len(b['accs']) for b in bins],
        'total_mbytes': [b['total'] for b in bins],
        'max_mbytes': [b['max'] for b in bins],
    }).with_columns(
        pl.lit(batch_name).alias('batch_name'),
        # Accessions in a pod are processed one at a time (streamed via FIFOs, no fastq
        # temp files), so peak local disk use is just the largest single run in the pod,
        # not the sum of all of them.
        (pl.col('max_mbytes') * storage_safety_factor + storage_headroom_mb)
            .cast(pl.Int64).alias('ephemeral_storage_mb'),
    ).select(
        'pod_accessions', 'batch_name', 'ephemeral_storage_mb',
        'pod_id', 'n_accessions', 'total_mbytes', 'max_mbytes',
    )
    return pods

In [6]:
def generate_and_write_batch(batch_name, to_process, blacklist_accs, num_acc_to_select,
                             making_batch=True, outdir='runlists_multi_sra', seed=42, **pack_kwargs):
    '''Choose accessions, pack them into pods, and write the pod CSV.

    Extra keyword args (e.g. max_accs_per_pod) are forwarded to pack_into_pods.
    '''
    os.makedirs(outdir, exist_ok=True)
    csv_path = os.path.join(outdir, f'{batch_name}.csv')

    if making_batch:
        possible = to_process.filter(~pl.col('acc').is_in(blacklist_accs))
        print(f"{possible.height} accessions available after blacklist of {len(blacklist_accs)}")
        chosen = possible.sample(min(num_acc_to_select, possible.height), seed=seed)
        pods = pack_into_pods(chosen, batch_name, **pack_kwargs)
        # randomise pod order so big and small pods are interleaved on submission
        pods = pods.sample(fraction=1, seed=seed, shuffle=True)
        pods.write_csv(csv_path)
        print(f"Wrote {pods.height} pods covering {pods['n_accessions'].sum()} accessions to {csv_path}")

    return pl.read_csv(csv_path)

### batch1 - a real batch of many small/medium runs

Adjust `num_acc_to_select` to control the batch size.

In [ ]:
batch1 = generate_and_write_batch(
    batch_name='multi_batch1',
    to_process=to_process,
    blacklist_accs=[],
    num_acc_to_select=10,
    making_batch=True,
    max_accs_per_pod=4,
)
print(batch1.shape)
print(batch1.select('n_accessions', 'total_mbytes', 'max_mbytes', 'ephemeral_storage_mb').describe())
batch1.head()

1083378 accessions available after blacklist of 0
10 runs to pack after dropping those > 30000 mbytes (0 dropped)
Wrote 3 pods covering 10 accessions to runlists_multi_sra/multi_batch1.csv
(3, 6)
shape: (9, 4)
┌────────────┬──────────────┬──────────────┬──────────────────────┐
│ statistic  ┆ n_accessions ┆ total_mbytes ┆ ephemeral_storage_mb │
│ ---        ┆ ---          ┆ ---          ┆ ---                  │
│ str        ┆ f64          ┆ f64          ┆ f64                  │
╞════════════╪══════════════╪══════════════╪══════════════════════╡
│ count      ┆ 3.0          ┆ 3.0          ┆ 3.0                  │
│ null_count ┆ 0.0          ┆ 0.0          ┆ 0.0                  │
│ mean       ┆ 3.333333     ┆ 7321.0       ┆ 14517.0              │
│ std        ┆ 1.154701     ┆ 9075.947278  ┆ 11798.826086         │
│ min        ┆ 2.0          ┆ 428.0        ┆ 5556.0               │
│ 25%        ┆ 4.0          ┆ 3931.0       ┆ 10110.0              │
│ 50%        ┆ 4.0          ┆ 3931.0      

pod_accessions,batch_name,ephemeral_storage_mb,pod_id,n_accessions,total_mbytes
str,str,i64,i64,i64,i64
"""ERR13867884 SRR24084207""","""multi_batch1""",5556,2,2,428
"""SRR11073009 SRR23994851 SRR313…","""multi_batch1""",10110,1,4,3931
"""DRR806302 ERR10688610 SRR35931…","""multi_batch1""",27885,0,4,17604


## Submit

Each CSV row becomes one Argo workflow. The `.sylref` reference DB is the same for
the whole batch and is passed as one URI (node-cached, not per pod). For a quick
one-off, substitute the columns into `sra_multi_workflow_template.yaml` and `argo submit`:

```
sed -e "s|{{workflow.parameters.SRA_accession_num}}|$pod_accessions|" \
    -e "s|{{workflow.parameters.batch_name}}|$batch_name|" \
    -e "s|{{workflow.parameters.ephemeral_storage_mb}}|$ephemeral_storage_mb|" \
    -e "s|{{workflow.parameters.reference_s3_uri}}|s3://woodcrob-sandpiper-us-east-1/db/r232.top50000.100.with_genomes.sylref|" \
    sra_multi_workflow_template.yaml | argo submit -n argo -
```

For throttled, resumable submission use `slow_argo_submission_multi.py`, which reads the CSV
and replaces the same placeholders per pod (with `--blacklist` to drop already-finished
accessions):

```
./slow_argo_submission_multi.py \
    --input-runlist-csv runlists_multi_sra/multi_batch1.csv \
    --workflow-template sra_multi_workflow_template.yaml \
    --reference-s3-uri s3://woodcrob-sandpiper-us-east-1/db/r232.top50000.100.with_genomes.sylref \
    --batch-size 200 --min-running-pending-file min_job_count
```

In [ ]:
# 5 small runs (small mbytes so a test download is quick), forced across 2 pods.
test_candidates = to_process.filter((pl.col('mbytes') >= 20) & (pl.col('mbytes') <= 300))
test5 = generate_and_write_batch(
    batch_name='multi_test5',
    to_process=test_candidates,
    blacklist_accs=[],
    num_acc_to_select=5,
    making_batch=True,
    max_accs_per_pod=3,   # 5 accessions -> 2 pods (3 + 2)
)
show_all(test5)

## Submit

Each CSV row becomes one Argo workflow. For a quick one-off, substitute the columns into
`sra_multi_workflow_template.yaml` and `argo submit`, e.g. per row:

```
sed -e "s|{{workflow.parameters.SRA_accession_num}}|$pod_accessions|" \
    -e "s|{{workflow.parameters.batch_name}}|$batch_name|" \
    -e "s|{{workflow.parameters.ephemeral_storage_mb}}|$ephemeral_storage_mb|" \
    sra_multi_workflow_template.yaml | argo submit -n argo -
```

For throttled, resumable submission use `slow_argo_submission_multi.py`, which reads the CSV
and replaces the same three placeholders per pod (with `--blacklist` to drop already-finished
accessions):

```
./slow_argo_submission_multi.py \
    --input-runlist-csv runlists_multi_sra/multi_batch1.csv \
    --workflow-template sra_multi_workflow_template.yaml \
    --batch-size 200 --min-running-pending-file min_job_count
```